<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/supervised/01_linear_regression_1d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S1 1変数の線形回帰(notebook)

本文:[`01-linear-regression-1d.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/supervised/01-linear-regression-1d.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-8):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

def make_data(N, sigma, rng, x_low=0.0, x_high=5.0, w_true=(1.0, 2.0)):
    # y = w0 + w1 x + 雑音(平均0・標準偏差 sigma)のデータを N 個作る
    x = rng.uniform(x_low, x_high, size=N)
    y = w_true[0] + w_true[1] * x + rng.normal(0.0, sigma, size=N)
    return x, y

## 1. 予測・残差・損失

**実装**:予測 $\hat y = w_0 + w_1 x$ を返す関数 `predict` と、平均二乗誤差を返す関数 `mse` を書く。

In [ ]:
def predict(w, x):
    w0, w1 = w
    raise NotImplementedError("TODO: 予測 w0 + w1 x(x は配列)")

def mse(w, x, y):
    raise NotImplementedError("TODO: 残差 y - 予測 の2乗の平均")

x3 = np.array([0.0, 1.0, 2.0]); y3 = np.array([1.0, 2.0, 2.0])
check("予測", predict((1.0, 0.5), x3), [1.0, 1.5, 2.0])
check("MSE", mse((1.0, 0.5), x3, y3), (0 + 0.25 + 0) / 3)

## 2. 損失を最小にする重み

**手計算**:3点 $(0,1), (1,2), (2,2)$ について、本文 §2 の式で $w_0, w_1$ を求める。

- 手計算:(ここに書く)

**実装**:本文 §2 の式で重みを求める関数 `fit_closed_form` を書く。

In [ ]:
def fit_closed_form(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    raise NotImplementedError("TODO: w1 = Σ(x-x̄)(y-ȳ) / Σ(x-x̄)^2、 w0 = ȳ - w1 x̄")
    return np.array([w0, w1])

check("3点の例(M4 と同じ答え)", fit_closed_form(x3, y3), [7 / 6, 1 / 2])
rng = np.random.default_rng(0)
x, y = make_data(30, 1.0, rng)
w_hat = fit_closed_form(x, y)
check("np.polyfit と一致", w_hat, np.polyfit(x, y, 1)[::-1])
check("直線は平均の点を通る", predict(w_hat, x.mean()), y.mean())
print("推定した重み:", np.round(w_hat, 3), "  本当の重み: [1. 2.]")

In [ ]:
xx = np.linspace(0, 5, 50)
plt.figure(figsize=(5, 3.5))
plt.plot(x, y, "o", alpha=0.6, label="data")
plt.plot(xx, predict(w_hat, xx), label="fitted line")
plt.plot(xx, 1 + 2 * xx, "k--", label="true line")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

## 3. 損失の形を見る

$L(w_0, w_1)$ の等高線を描く。求めた重み(×)が楕円の中心にあることを確かめる。

**確かめる**:求めた重みを少しずらすと、損失は必ず増えるはずである。

In [ ]:
W0, W1 = np.meshgrid(np.linspace(-1, 3, 120), np.linspace(1, 3, 120))
L = np.array([[mse((a, b), x, y) for a, b in zip(r0, r1)] for r0, r1 in zip(W0, W1)])
plt.figure(figsize=(5, 4))
cs = plt.contour(W0, W1, L, levels=np.quantile(L, [0.01, 0.03, 0.1, 0.2, 0.4, 0.7]))
plt.clabel(cs, fontsize=7)
plt.plot(*w_hat, "rx", ms=10, mew=2, label="closed form")
plt.xlabel("w0"); plt.ylabel("w1"); plt.legend(); plt.show()

rng2 = np.random.default_rng(1)
print("ずらすと必ず増えるか:",
      all(mse(w_hat + 0.05 * rng2.normal(size=2), x, y) > mse(w_hat, x, y) for _ in range(100)))

## 4. 推定のばらつき

データを作り直すたびに、推定した重みは変わる。

**予想**:データの数 $N$ を 10 → 40 → 160 と4倍ずつにすると、$\hat w_1$ の標準偏差はどうなるか。

- 予想:(ここに書く)

**実装**:seed を変えて `n_repeat` 回データを作り、そのたびに $\hat w_1$ を求めて、その標準偏差を返す関数を書く。

In [ ]:
def w1_spread(N, sigma, n_repeat=500, x_low=0.0, x_high=5.0):
    w1s = []
    raise NotImplementedError("TODO: seed を 0, 1, ..., n_repeat-1 と変えてデータを作り、fit_closed_form の w1 を集める")
    return np.std(w1s)

for N in [10, 40, 160]:
    print(f"N={N:4d}, σ=1.0: w1 の標準偏差 {w1_spread(N, 1.0):.3f}")
print()
for sigma in [0.5, 1.0, 2.0]:
    print(f"N=  40, σ={sigma}: w1 の標準偏差 {w1_spread(40, sigma):.3f}")

$N$ を4倍にすると標準偏差はおよそ半分、$\sigma$ を2倍にすると標準偏差はおよそ2倍になる。

**x の範囲の影響**:同じ $N = 40$、$\sigma = 1$ で、$x$ を $[0, 1]$ から集めた場合と $[0, 5]$ から集めた場合を比べる。

In [ ]:
print("x を [0, 1] から集めた: ", round(w1_spread(40, 1.0, x_low=0.0, x_high=1.0), 3))
print("x を [0, 5] から集めた: ", round(w1_spread(40, 1.0, x_low=0.0, x_high=5.0), 3))

## 5. 学習用と評価用のデータを分ける

データを学習用(70%)と評価用(30%)に分け、学習用だけで重みを決めて、両方の MSE を測る。

**予想**:学習誤差とテスト誤差は、それぞれおよそいくつになるか($\sigma = 1$)。

- 予想:(ここに書く)

**実装**:データの順番をランダムに並べかえて、学習用と評価用に分ける関数 `split` を書く。

In [ ]:
def split(x, y, test_ratio, rng):
    n = len(x)
    n_test = int(round(n * test_ratio))
    raise NotImplementedError("TODO: 添字をランダムに並べかえ(rng.permutation)、最初の n_test 個を評価用、残りを学習用にする")
    return x[tr], y[tr], x[te], y[te]

rng = np.random.default_rng(3)
x, y = make_data(100, 1.0, rng)
x_tr, y_tr, x_te, y_te = split(x, y, 0.3, rng)
check("個数", [len(x_tr), len(x_te)], [70, 30])
check("重なりがない", len(np.intersect1d(x_tr, x_te)), 0)
w = fit_closed_form(x_tr, y_tr)
print(f"学習誤差 {mse(w, x_tr, y_tr):.3f}   テスト誤差 {mse(w, x_te, y_te):.3f}   (σ^2 = 1.0)")

1回だけでは、たまたまの差かどうか分からない。データの作り方と分け方(seed)を変えて2000回くり返し、平均をとる。

In [ ]:
tr_err, te_err = [], []
for seed in range(2000):
    r = np.random.default_rng(seed)
    xs, ys = make_data(100, 1.0, r)
    a, b, c, d = split(xs, ys, 0.3, r)
    w = fit_closed_form(a, b)
    tr_err.append(mse(w, a, b)); te_err.append(mse(w, c, d))
print(f"学習誤差の平均 {np.mean(tr_err):.3f}   テスト誤差の平均 {np.mean(te_err):.3f}")

学習誤差は平均すると $\sigma^2 = 1$ より少し小さく(理論的には $\sigma^2(1 - 2/70) \approx 0.97$)、テスト誤差は少し大きい。モデルが学習用のデータに合わせて重みを選んでいるためである。差はわずかなので、くり返しの回数が少ないと逆転して見えることもある(50回で試してみるとよい)。この差は、モデルを複雑にすると大きくなる(S4 の過学習)。

## 6. 条件を変える

1. 4 で、$N = 3$ のときの $\hat w_1$ のばらつきを調べる。少ないデータで求めた傾きは、どのくらい信用できるか
2. 5 で、正解を $y = 1 + 2x + 0.5x^2 + \varepsilon$(曲線)にすると、直線モデルのテスト誤差はどうなるか。$\sigma^2$ より大きくなるのはなぜか
3. 5 で、評価用の割合を 0.3 から 0.05 にすると、テスト誤差のばらつきはどう変わるか

## 7. 説明(自分の言葉で書く)

1. 残差と損失の違いは何か
   - (ここに書く)
2. 推定した重みが、データを作り直すたびに変わるのはなぜか。ばらつきを小さくするにはどうすればよいか
   - (ここに書く)
3. 学習誤差だけでモデルを評価してはいけないのはなぜか
   - (ここに書く)